# MultiCaRe filter for the A&E triage project

Selects the images (and their linked case reports) from **OpenMed/multicare-images** that fit the project:
an A&E doctor's view of an adult patient who came to the emergency department, with imaging an A&E team would order
(X-ray, CT, ultrasound) or a photo of an acute problem, a reusable licence, and vital signs in the case text.

All rules live in `../src/multicare_filter.py` (`CONFIG`), so any rule can be loosened or tightened and re-run.
This folder is independent of `vitals/`: neither imports the other.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import multicare_filter as mf

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 140)
OUT = Path.cwd().parent / "output"
OUT.mkdir(exist_ok=True)
mf.CONFIG

{'licenses': ['CC0', 'CC BY', 'CC BY-SA', 'CC BY-NC', 'CC BY-NC-SA'],
 'radiology_subtypes': ['x_ray', 'ct', 'ultrasound'],
 'photo_subtypes': ['skin_photograph', 'other_medical_photograph'],
 'include_ecg': False,
 'min_caption_chars': 20,
 'exclude_later_timepoints': True,
 'exclude_echo': False,
 'min_age': 16,
 'require_ed': True,
 'min_vitals': 1}

## 1. Load (from the local Hugging Face cache, image bytes not loaded)

In [2]:
img = mf.load_images()
cases = mf.annotate_cases(mf.load_cases())
print(f"{len(img):,} images, {len(cases):,} cases")

161,702 images, 110,182 cases


## 2. Filter funnel

In [3]:
sel, funnel = mf.apply_filters(img, cases)
funnel

,step,images,cases,articles
0,all images,161702,42044,38832
1,licence allows reuse,121710,30972,28561
2,"A&E modality (X-ray/CT/US, acute photo)",43099,17071,16069
3,caption >= 20 chars,41738,16922,15941
4,"not a later-stage image (post-op, follow-up, healed)",37663,16354,15442
5,linked to a case,37663,16354,15442
6,age >= 16,30762,13208,12565
7,presented to emergency department,5492,2374,2314
8,>= 1 measured vital sign(s) in text,2340,1015,989


## 3. What was selected

In [4]:
by_case = sel.drop_duplicates("case_id")
summary = {
    "modality (images)": sel.modality.value_counts(),
    "body region (radiology images)": sel.radiology_region.value_counts(),
    "licence (images)": sel.license.value_counts(),
    "department guess (cases)": by_case.department_guess.value_counts(),
}
for name, s in summary.items():
    print(f"--- {name}"); print(s.to_string(), "\n")

--- modality (images)
modality
ct             1339
x_ray           513
ultrasound      396
acute_photo      92 

--- body region (radiology images)
radiology_region
thorax        1147
abdomen        663
head           232
pelvis          72
lower_limb      54
neck            37
upper_limb      30
breast          12
whole_body       1 

--- licence (images)
license
CC BY          1255
CC BY-NC        738
CC BY-NC-SA     347 

--- department guess (cases)
department_guess
Respiratory      260
Cardiology       219
Surgery          193
Other/unclear    155
Neurology        143
Orthopaedics      45 



In [5]:
vit = pd.DataFrame({
    "cases mentioning it": by_case[[f"v_{k}" for k in mf.VITALS]].sum().values,
    "% of selected cases": (by_case[[f"v_{k}" for k in mf.VITALS]].mean() * 100).round(1).values,
}, index=list(mf.VITALS))
print("vital signs mentioned per case:", by_case.n_vitals.value_counts().sort_index().to_dict())
vit

vital signs mentioned per case: {1: 228, 2: 224, 3: 212, 4: 225, 5: 126}


,cases mentioning it,% of selected cases
blood_pressure,800,78.8
heart_rate,788,77.6
resp_rate,431,42.5
oxygen_sat,403,39.7
temperature,420,41.4


In [6]:
print(f"cases: {len(by_case)}, images per case: median {sel.groupby('case_id').size().median():.0f}, "
      f"max {sel.groupby('case_id').size().max()}")
print(f"age: median {by_case.age.median():.0f} (range {by_case.age.min():.0f}-{by_case.age.max():.0f}); "
      f"gender: {by_case.gender.value_counts().to_dict()}")
print(f"department tied between two candidates: {by_case.department_tied.sum()} cases")
print(f"echocardiograms: {sel.is_echo.sum()} of {(sel.modality == 'ultrasound').sum()} ultrasound images "
      f"({sel.is_echo.sum()} of {len(sel)} images overall)")

cases: 1015, images per case: median 2, max 16
age: median 50 (range 16-100); gender: {'Male': 585, 'Female': 426, 'Unknown': 4}
department tied between two candidates: 97 cases
echocardiograms: 207 of 396 ultrasound images (207 of 2340 images overall)


## 4. Stricter and looser options

Same pipeline with different `CONFIG` values, so the group can pick the trade-off between size and quality.

In [7]:
options = {
    "default (>=1 vital)": {},
    ">= 3 vitals (enough for a meaningful NEWS2)": {"min_vitals": 3},
    "+ ECG images": {"include_ecg": True},
    "+ MRI": {"radiology_subtypes": ["x_ray", "ct", "ultrasound", "mri"]},
    "CC BY only (most permissive licence)": {"licenses": ["CC0", "CC BY"]},
    "exclude echocardiograms": {"exclude_echo": True},
    "keep later-stage images": {"exclude_later_timepoints": False},
    "no ED requirement": {"require_ed": False},
}
rows = []
for name, change in options.items():
    s, _ = mf.apply_filters(img, cases, {**mf.CONFIG, **change})
    rows.append({"option": name, "images": len(s), "cases": s.case_id.nunique()})
pd.DataFrame(rows)

,option,images,cases
0,default (>=1 vital),2340,1015
1,>= 3 vitals (enough for a meaningful NEWS2),1251,563
2,+ ECG images,2624,1111
3,+ MRI,2776,1104
4,CC BY only (most permissive licence),1255,536
5,exclude echocardiograms,2133,967
6,keep later-stage images,2506,1037
7,no ED requirement,6824,2903


## 5. Example images

In [ ]:
rng = np.random.default_rng(3)
picks = []
for m in ["x_ray", "ct", "ultrasound", "acute_photo"]:
    pool = sel[sel.modality == m]
    picks += list(pool.sample(min(3, len(pool)), random_state=3).itertuples())
images = mf.load_image_bytes([p.file_id for p in picks])
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, p in zip(axes.T.flat, picks):
    ax.imshow(images[p.file_id].convert("RGB"))
    ax.set_title(f"{p.modality} | {p.department_guess}\n{p.caption[:60]}...", fontsize=8, loc="left")
    ax.axis("off")
fig.tight_layout()
plt.show()

## 6. Save

Metadata only (no image bytes). Load the images later from the Hugging Face dataset by `file_id`.

In [9]:
img_cols = ["file_id", "file", "main_image", "image_component", "patient_id", "case_id", "article_id",
            "license", "modality", "image_type", "image_subtype", "radiology_region", "radiology_region_granular",
            "radiology_view", "is_echo", "caption"]
case_cols = ["case_id", "article_id", "age", "gender", "department_guess", "department_tied", "n_vitals",
             *[f"v_{k}" for k in mf.VITALS], "case_text"]
sel[img_cols].to_csv(OUT / "selected_images.csv", index=False, encoding="utf-8-sig")
by_case[case_cols].to_csv(OUT / "selected_cases.csv", index=False, encoding="utf-8-sig")
funnel.to_csv(OUT / "funnel.csv", index=False)

# Hand-check sheet for the department labels: 40 random cases, fill in 'correct_department'
check = by_case.sample(40, random_state=0)[["case_id", "department_guess", "department_tied", "case_text"]].copy()
check["case_text"] = check.case_text.str[:700]
check["correct_department"] = ""
check.to_csv(OUT / "department_hand_check.csv", index=False, encoding="utf-8-sig")
print(sorted(p.name for p in OUT.iterdir()))

['department_hand_check.csv', 'funnel.csv', 'selected_cases.csv', 'selected_images.csv']


## Summary

(Run of 19 Oct 2026, default `CONFIG`.)

- **Selected: 2,340 images from 1,015 cases** (989 articles), out of 161,702 images. The ED-presentation rule is the
  biggest cut (30,762 → 5,492 images), then the vital-signs rule (→ 2,340).
- **Modality:** CT 1,339, X-ray 513, ultrasound 396 (207 of them echocardiograms), acute photos 92.
  **Regions:** mostly thorax (1,147) and abdomen (663), then head (232). Limb images are scarce (84), so
  orthopaedic cases are under-represented (45 cases).
- **Department guess (cases):** Respiratory 260, Cardiology 219, Surgery 193, Neurology 143, Orthopaedics 45,
  Other/unclear 155; 97 cases are tied between two departments. This is a keyword guess from the case text and must
  be hand-checked (`output/department_hand_check.csv`).
- **Vital signs in the text:** BP 79% of cases, heart rate 78%, RR 43%, SpO2 40%, temperature 41%; 563 cases mention
  ≥ 3 vitals. The filter only detects that vitals are *mentioned*; extracting the numbers is a separate step.
- **Licences:** CC BY 1,255, CC BY-NC 738, CC BY-NC-SA 347 images (ND and unlicensed excluded).
- **Known limits:** the rules are keyword-based, so some irrelevant images remain (e.g. a liver CT in a case labelled
  Neurology, because department is per case, not per image), and some later-stage images may slip past the timepoint
  rule. Spot-check before training.